# 🤖 GirishOS Backend - Google Colab Setup

This notebook runs the GirishOS backend on Google Colab with free GPU.

## Prerequisites:
1. A photo of Girish (front-facing, neutral expression, 512x512+)
2. A free Groq API key from https://groq.com
3. A free ngrok auth token from https://ngrok.com

## How it connects to your frontend:
1. This notebook starts a FastAPI server on Colab
2. ngrok creates a public URL that tunnels to this server
3. Your Next.js frontend connects to this URL via WebSocket
4. Questions flow: Frontend → ngrok → Colab Backend → Pipeline → Video → Frontend

In [2]:
# Cell 2: Install all dependencies
!pip install -q fastapi uvicorn[standard] edge-tts groq pyngrok nest-asyncio websockets python-multipart aiofiles

# Clone SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git 2>/dev/null || echo "Already cloned"
%cd SadTalker
!pip install -q -r requirements.txt
!bash scripts/download_models.sh 2>/dev/null || echo "Models already downloaded"
%cd ..

print("✅ All dependencies installed!")


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


"Already cloned"
[WinError 2] The system cannot find the file specified: 'SadTalker'
d:\MyProjects


The system cannot find the path specified.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


"Models already downloaded"
d:\
✅ All dependencies installed!


The system cannot find the path specified.


In [3]:
# Cell 3: Upload Girish's photo
import os
from google.colab import files

os.makedirs("assets/photos", exist_ok=True)

print("📸 Upload Girish's photo (PNG or JPG, front-facing, neutral expression)")
print("   Requirements: 512x512+ pixels, good lighting, plain background")
print("")

uploaded = files.upload()

# Move uploaded file to assets
for filename in uploaded.keys():
    os.rename(filename, f"assets/photos/{filename}")
    print(f"✅ Photo saved as: assets/photos/{filename}")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Cell 4: Configuration
import os
from google.colab import userdata

# Set your Groq API key (stored in Colab Secrets)
# Go to: Left panel → 🔑 Secrets → Add "GROQ_API_KEY"
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("✅ Groq API key loaded from Colab Secrets")
except Exception:
    GROQ_API_KEY = input("Enter your Groq API key: ")

# Set ngrok auth token
# Go to: Left panel → 🔑 Secrets → Add "NGROK_AUTH_TOKEN"
try:
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
    print("✅ ngrok auth token loaded from Colab Secrets")
except Exception:
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")

# Find the photo
photo_path = None
for f in os.listdir("assets/photos"):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
        photo_path = f"assets/photos/{f}"
        break

if photo_path:
    print(f"✅ Using photo: {photo_path}")
else:
    print("❌ No photo found! Please run Cell 3 to upload.")

print("\n🎯 Configuration complete!")

In [ ]:
# Cell 5: Start the GirishOS backend server
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start tunnel
public_url = ngrok.connect(8000, "http")
ws_url = str(public_url).replace("https://", "wss://").replace("http://", "ws://")

print("=" * 60)
print("🚀 GirishOS Backend is LIVE!")
print("=" * 60)
print(f"")
print(f"📡 HTTP URL:      {public_url}")
print(f"🔌 WebSocket URL: {ws_url}/ws")
print(f"")
print(f"📋 Set this in your frontend .env file:")
print(f"   NEXT_PUBLIC_BACKEND_URL={ws_url}")
print(f"")
print("=" * 60)

# Start FastAPI server (this will block - keep this cell running)
import uvicorn
# TODO: Import actual app once pipeline is implemented
# For now, start with placeholder
from backend.main import app
uvicorn.run(app, host="0.0.0.0", port=8000)